In [1]:
#!pip install mistralai pandas tqdm

In [2]:
from mistralai import Mistral
import json
import re
import time
from tqdm import tqdm

In [3]:
PROMPT_TEMPLATE = """
You are a biology expert.

For each text, extract biological taxons.

Return ONLY valid JSON.
Format:
[
  ["taxon1", "taxon2"],
  null
]

Texts:
{texts}
"""


In [4]:
client = Mistral(api_key="1nCRukeZXLfBrLYxFpnoVhSYcIXLvTnq")

In [5]:
def call_mistral_with_retry(messages, max_retries=5):
    delay = 2

    for attempt in range(max_retries):
        try:
            response = client.chat.complete(
                model="mistral-medium-latest",
                messages=messages,
                temperature=0,
                response_format={"type": "json_object"}
            )
            return response

        except Exception as e:
            if "429" in str(e):
                print(f"Rate limit → sleeping {delay}s...")
                time.sleep(delay)
                delay *= 2  # exponential backoff
            else:
                raise e

    raise RuntimeError("Too many retries")


In [6]:
def batch_extract(texts, batch_size=20, sleep_between=0.5):
    results = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        formatted = "\n".join(
            [f"{idx}. {t}" for idx, t in enumerate(batch)]
        )

        prompt = PROMPT_TEMPLATE.format(texts=formatted)

        messages = [{"role": "user", "content": prompt}]

        response = call_mistral_with_retry(messages)

        data = json.loads(response.choices[0].message.content)

        results.extend(data)

        time.sleep(sleep_between)  # évite burst

    return results


In [10]:
def batch_extract(texts, batch_size=20, sleep_between=0.5):
    results = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        formatted = "\n".join(
            [f"{idx}. {t}" for idx, t in enumerate(batch)]
        )
        prompt = f"""
You are a biology expert.

For EACH text, extract taxons.

Return ONLY a JSON ARRAY (not object).
The array must contain EXACTLY {len(batch)} elements.
Each element:
- list of taxon names
- or null

Example:
[
  ["Homo sapiens"],
  null,
  ["Canis lupus"]
]

Texts:
{formatted}
"""
        messages = [{"role": "user", "content": prompt}]

        response = call_mistral_with_retry(messages)

        data = json.loads(response.choices[0].message.content)

        if not isinstance(data, list):
            data = [None] * len(batch)

        if len(data) < len(batch):
            data += [None] * (len(batch) - len(data))

        if len(data) > len(batch):
            data = data[:len(batch)]

        results.extend(data)

        time.sleep(sleep_between)

    return results


In [11]:
import pandas as pd

df=pd.read_csv('Data.csv',delimiter=',',on_bad_lines="skip")

texts = (df["Title"] + " " + df["Description"]).tolist()

df["taxons"] = batch_extract(texts, batch_size=20)

df.to_csv("output2.csv", index=False)
